# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [74]:
# Load the libraries as required.
import pandas as pd
import numpy as np
import pickle
import shap
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score

In [75]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area'
]
fires_dt = pd.read_csv('../../05_src/data/fires/forestfires.csv', header=0, names=columns)
X = fires_dt.drop('area', axis=1)
y = fires_dt['area']

# Get X and Y

Create the features data frame and target data.

In [76]:
numeric_features = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
categorical_features = ['month', 'day']

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [77]:
preproc1 = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [78]:
log_features = ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'wind', 'rain']
other_numeric = [f for f in numeric_features if f not in log_features]
preproc2 = ColumnTransformer([
    ('num_log', Pipeline([
        ('scaler', MinMaxScaler()),
        ('log', FunctionTransformer(np.log1p, validate=False))
    ]), log_features),
    ('num_other', StandardScaler(), other_numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [79]:
# Pipeline A = preproc1 + baseline
pipe_a = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', LinearRegression())
])

In [80]:
# Pipeline B = preproc2 + baseline
pipe_b = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', LinearRegression())
])

In [81]:
# Pipeline C = preproc1 + advanced model
pipe_c = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', RandomForestRegressor(random_state=42))
])

In [82]:
# Pipeline D = preproc2 + advanced model
pipe_d = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', RandomForestRegressor(random_state=42))
])
    

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [83]:
param_grid_rf = {'regressor__n_estimators': [50, 100, 200, 300]}

grid_c = GridSearchCV(pipe_c, param_grid_rf, cv=5, scoring='neg_root_mean_squared_error')
grid_d = GridSearchCV(pipe_d, param_grid_rf, cv=5, scoring='neg_root_mean_squared_error')

In [84]:
pipe_a.fit(X, y)
pipe_b.fit(X, y)
grid_c.fit(X, y)
grid_d.fit(X, y)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num_log',
                                                                         Pipeline(steps=[('scaler',
                                                                                          MinMaxScaler()),
                                                                                         ('log',
                                                                                          FunctionTransformer(func=<ufunc 'log1p'>))]),
                                                                         ['ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('num_other',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'rh']),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor',
                                        RandomForestRegressor(random_state=42))]),
             param_grid={'regressor__n_estimators': [50, 100, 200, 300]},
             scoring='neg_root_mean_squared_error')

In [85]:
models = {
    "LinearRegression + preproc1": pipe_a,
    "LinearRegression + preproc2": pipe_b,
    "RF + preproc1": grid_c.best_estimator_,
    "RF + preproc2": grid_d.best_estimator_
}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error')
    print(f"{name} CV RMSE: {-scores.mean():.3f} ± {scores.std():.3f}")

LinearRegression + preproc1 CV RMSE: 54.648 ± 37.991
LinearRegression + preproc2 CV RMSE: 54.543 ± 38.092
RF + preproc1 CV RMSE: 59.046 ± 35.691
RF + preproc2 CV RMSE: 64.524 ± 32.684


# Evaluate

+ Which model has the best performance?

The best model is Pipeline B (Linear Regression + preproc2) since the CV RMSE is the lowest - meaning it has the lowest ratio of the root mean square error (RMSE) to the mean of the target variable (area).

# Export

+ Save the best performing model to a pickle file.

In [86]:
best_model = pipe_a  # or whichever has lowest RMSE
with open('best_forestfire_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [87]:
import shap
X_transformed = best_model.named_steps['preprocessing'].transform(X)
feature_names = best_model.named_steps['preprocessing'].get_feature_names_out()
explainer = shap.Explainer(best_model.named_steps['regressor'], X_transformed, feature_names=feature_names)
shap_values = explainer(X_transformed)
obs_idx = 0
shap_df = pd.DataFrame({
    'feature': feature_names,
    'shap_value': shap_values[obs_idx].values
})
print(shap_df)
all_shap_df = pd.DataFrame(shap_values.values, columns=feature_names)
mean_abs_shap = all_shap_df.abs().mean().sort_values(ascending=False)
print(mean_abs_shap)

           feature  shap_value
0     num__coord_x    4.647016
1     num__coord_y   -0.093020
2        num__ffmc    0.417104
3         num__dmc  -16.327543
4          num__dc   61.646275
5         num__isi    1.994903
6        num__temp  -13.896549
7          num__rh   -0.772176
8        num__wind    5.172599
9        num__rain    0.045047
10  cat__month_apr    0.247571
11  cat__month_aug   -7.522498
12  cat__month_dec   -0.226212
13  cat__month_feb    1.341704
14  cat__month_jan   -0.000000
15  cat__month_jul   -0.206092
16  cat__month_jun    0.541247
17  cat__month_mar  -26.951038
18  cat__month_may   -0.000000
19  cat__month_nov   -0.000000
20  cat__month_oct   -1.326549
21  cat__month_sep  -19.100197
22    cat__day_fri   -6.186976
23    cat__day_mon    0.183585
24    cat__day_sat   -2.001187
25    cat__day_sun    0.523665
26    cat__day_thu   -0.162694
27    cat__day_tue   -0.015458
28    cat__day_wed    0.302275
num__dc           24.996994
cat__month_sep    22.684741
cat__month_aug

For the first observation in the test set, the SHAP values for DC are the highest. It is also a strong predictor overall for the dataset. The feature with low SHAP importance, that could be removed from the training set, is day of the week. This feature could probably be removed from the model.

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.